# 분리수거 판별기 — CNN 전이학습 (Colab)

EfficientNet-B0 전이학습으로 쓰레기 재질 4클래스(plastic/can/glass/paper) 분류 모델을 학습한다.

**사용법**
1. 로컬 `data/trashnet/` 의 4개 클래스 폴더(`can`, `glass`, `paper`, `plastic`)를 그대로
   Google Drive의 `MyDrive/trash-check/dataset/` 아래에 업로드 (zip 압축 불필요)
2. Colab 런타임 유형을 **GPU(T4)** 로 변경
3. 아래 Drive 마운트 셀부터 전체 실행 (Drive의 데이터를 Colab 로컬 디스크로 복사 후 학습 — I/O 속도 확보)
4. 학습 완료 후 `best_model.pt`가 `MyDrive/trash-check/`에 저장됨 → 로컬 `model/best_model.pt`로 다운로드해 배치

데이터셋 출처: TrashNet (Gary Thung & Mindy Yang, https://github.com/garythung/trashnet)

In [ ]:
# 1. Google Drive 마운트 후 데이터셋을 Colab 로컬 디스크로 복사 (학습 I/O 속도 확보)
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/trash-check'          # 산출물(best_model.pt, reports/) 저장 위치
DRIVE_DATASET_DIR = os.path.join(DRIVE_DIR, 'dataset')     # 클래스 폴더가 여기 바로 아래 있어야 함

# 5클래스: trash(일반쓰레기) 포함 — 이물질/판별불가 항목의 목적지 클래스
EXPECTED_CLASSES = {'can', 'glass', 'paper', 'plastic', 'trash'}
assert os.path.isdir(DRIVE_DATASET_DIR), f'경로를 찾을 수 없음: {DRIVE_DATASET_DIR}'
found = set(os.listdir(DRIVE_DATASET_DIR))
assert EXPECTED_CLASSES.issubset(found), (
    f'{DRIVE_DATASET_DIR} 아래에서 {sorted(EXPECTED_CLASSES)} 폴더를 찾지 못했습니다. '
    f'실제 목록: {sorted(found)}'
)

DATA_DIR = '/content/data/trashnet'
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
shutil.copytree(DRIVE_DATASET_DIR, DATA_DIR)

# macOS 리소스 포크 메타데이터 파일(._*, .DS_Store) 등 이미지가 아닌 파일 제거
removed = 0
for root, dirs, files in os.walk(DATA_DIR):
    for name in files:
        if name.startswith('._') or name == '.DS_Store':
            os.remove(os.path.join(root, name))
            removed += 1
if removed:
    print(f'비이미지 메타데이터 파일 {removed}개 제거')

print('DATA_DIR:', DATA_DIR)
print(os.listdir(DATA_DIR))

In [ ]:
# 2. 데이터셋 구성 (train/val 8:2 분할, 증강 적용)
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

IMG_SIZE = 224
BATCH = 32
SEED = 42

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

full = datasets.ImageFolder(DATA_DIR)
CLASS_NAMES = full.classes  # ['can', 'glass', 'paper', 'plastic'] (알파벳순)
print('클래스:', CLASS_NAMES)

n_val = int(len(full) * 0.2)
g = torch.Generator().manual_seed(SEED)
train_idx, val_idx = random_split(range(len(full)), [len(full) - n_val, n_val], generator=g)

train_ds = torch.utils.data.Subset(datasets.ImageFolder(DATA_DIR, transform=train_tf), train_idx.indices)
val_ds = torch.utils.data.Subset(datasets.ImageFolder(DATA_DIR, transform=val_tf), val_idx.indices)

train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2)
val_dl = DataLoader(val_ds, batch_size=BATCH, num_workers=2)
print(f'train {len(train_ds)} / val {len(val_ds)}')

In [ ]:
# 3. 클래스 불균형 가중치 (paper가 타 클래스의 약 2배)
from collections import Counter

label_counts = Counter(full.targets[i] for i in train_idx.indices)
total = sum(label_counts.values())
class_weights = torch.tensor(
    [total / (len(CLASS_NAMES) * label_counts[i]) for i in range(len(CLASS_NAMES))],
    dtype=torch.float,
)
print(dict(zip(CLASS_NAMES, class_weights.tolist())))

In [ ]:
# 4. 모델: EfficientNet-B0 전이학습 (분류 헤드만 교체)
from torchvision import models
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(CLASS_NAMES))
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

In [ ]:
# 5. 학습 루프 (best val accuracy 체크포인트를 Drive에 직접 저장 — 세션 끊김 대비)
# 발표용 그래프를 위해 epoch별 지표를 history에 기록한다.
EPOCHS = 15
best_acc = 0.0
BEST_MODEL_PATH = os.path.join(DRIVE_DIR, 'best_model.pt')
history = {'train_loss': [], 'val_acc': []}

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    scheduler.step()

    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in val_dl:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
    val_acc = correct / len(val_ds)
    avg_train_loss = train_loss / len(train_ds)
    history['train_loss'].append(avg_train_loss)
    history['val_acc'].append(val_acc)

    print(f'epoch {epoch+1:02d} | train_loss {avg_train_loss:.4f} | val_acc {val_acc:.4f}')
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({'state_dict': model.state_dict(), 'classes': CLASS_NAMES}, BEST_MODEL_PATH)
        print(f'  -> best 갱신, Drive에 저장 (val_acc {best_acc:.4f})')

In [ ]:
# 6. 평가: 클래스별 리포트 + 혼동행렬
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

ckpt = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()

y_true, y_pred = [], []
with torch.no_grad():
    for x, y in val_dl:
        y_true.extend(y.tolist())
        y_pred.extend(model(x.to(device)).argmax(1).cpu().tolist())

report_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True)
cm = confusion_matrix(y_true, y_pred)

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
print(cm)

## 발표용 그래프 저장

아래 3개 셀은 `MyDrive/trash-check/reports/`에 PNG를 저장한다.
1. 학습 곡선 (train loss / val accuracy)
2. 혼동행렬 히트맵
3. 클래스별 Precision/Recall/F1 막대그래프

In [ ]:
import matplotlib.pyplot as plt

REPORTS_DIR = os.path.join(DRIVE_DIR, 'reports')
os.makedirs(REPORTS_DIR, exist_ok=True)

# (1) 학습 곡선: train loss / val accuracy
epochs_range = range(1, len(history['train_loss']) + 1)
fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(epochs_range, history['train_loss'], 'o-', color='tab:red', label='Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Train Loss', color='tab:red')
ax1.tick_params(axis='y', labelcolor='tab:red')

ax2 = ax1.twinx()
ax2.plot(epochs_range, history['val_acc'], 's-', color='tab:blue', label='Val Accuracy')
ax2.set_ylabel('Val Accuracy', color='tab:blue')
ax2.tick_params(axis='y', labelcolor='tab:blue')

plt.title('학습 곡선 (Train Loss / Val Accuracy)')
fig.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'training_curve.png'), dpi=150)
plt.show()

In [ ]:
# (2) 혼동행렬 히트맵
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASS_NAMES)))
ax.set_yticks(range(len(CLASS_NAMES)))
ax.set_xticklabels(CLASS_NAMES)
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('예측 클래스')
ax.set_ylabel('실제 클래스')
ax.set_title('혼동행렬 (Confusion Matrix)')

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')

fig.colorbar(im, ax=ax)
fig.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

In [ ]:
# (3) 클래스별 Precision/Recall/F1 막대그래프
metrics = ['precision', 'recall', 'f1-score']
x = np.arange(len(CLASS_NAMES))
width = 0.25

fig, ax = plt.subplots(figsize=(7, 4))
for i, m in enumerate(metrics):
    values = [report_dict[c][m] for c in CLASS_NAMES]
    ax.bar(x + (i - 1) * width, values, width, label=m)

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title(f"클래스별 Precision/Recall/F1 (전체 정확도 {report_dict['accuracy']:.2%})")
ax.legend()
fig.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'class_metrics.png'), dpi=150)
plt.show()

print(f'그래프 저장 완료: {REPORTS_DIR}')

In [ ]:
# 7. best_model.pt는 이미 Drive(`MyDrive/trash-check/best_model.pt`)에 저장됨
# 로컬 model/ 폴더에도 두려면 Drive에서 해당 파일을 다운로드해 배치
print(f'저장 위치: {BEST_MODEL_PATH}')